# Calorie Expenditure Model Training

Train a regression model for Kaggle Playground Series S5E5: Predict Calorie Expenditure.

Outputs saved to `/kaggle/working/calorie_expenditure_artifacts`:

- `calorie_expenditure_model.joblib`
- `metrics.json`
- `feature_schema.json`
- `submission.csv` when `test.csv` is available


In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET_COLUMN = "Calories"
ARTIFACT_DIR = Path("/kaggle/working/calorie_expenditure_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## Locate Kaggle Files

This searches common Kaggle input paths, so the notebook works whether the competition dataset is attached directly or copied into a different input folder.

In [ ]:
def find_dataset_file(filename: str) -> Path:
    candidates = list(Path("/kaggle/input").glob(f"**/{filename}"))
    if candidates:
        return candidates[0]
    local_candidate = Path(filename)
    if local_candidate.exists():
        return local_candidate
    raise FileNotFoundError(f"Could not find {filename}. Attach the Kaggle competition data first.")


train_path = find_dataset_file("train.csv")
test_path = find_dataset_file("test.csv") if list(Path("/kaggle/input").glob("**/test.csv")) else None

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path) if test_path else None

print(f"Train path: {train_path}")
print(f"Train shape: {train_df.shape}")
if test_df is not None:
    print(f"Test path: {test_path}")
    print(f"Test shape: {test_df.shape}")

train_df.head()


## Prepare Features

The competition target is expected to be `Calories`. The model ignores `id` as a feature and preserves it only for submission output.

In [ ]:
if TARGET_COLUMN not in train_df.columns:
    raise ValueError(f"Expected target column {TARGET_COLUMN!r}; found {train_df.columns.tolist()}")

id_columns = [column for column in ["id", "Id", "ID"] if column in train_df.columns]
feature_columns = [column for column in train_df.columns if column not in id_columns + [TARGET_COLUMN]]

X = train_df[feature_columns].copy()
y = train_df[TARGET_COLUMN].astype(float).clip(lower=0)

categorical_columns = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

feature_schema = {
    "target_column": TARGET_COLUMN,
    "id_columns": id_columns,
    "feature_columns": feature_columns,
    "numeric_columns": numeric_columns,
    "categorical_columns": categorical_columns,
}

feature_schema


## Train Baseline Models

The target is trained through `log1p` and restored with `expm1`, which usually behaves better for calorie-style positive regression targets and aligns with RMSLE-style evaluation.

In [ ]:
def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.maximum(np.asarray(y_true, dtype=float), 0)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def build_model(regressor):
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )
    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

candidate_models = {
    "hist_gradient_boosting": build_model(
        HistGradientBoostingRegressor(
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.02,
            random_state=RANDOM_STATE,
        )
    ),
    "random_forest": build_model(
        RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ),
}

results = {}
trained_models = {}

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    valid_pred = np.maximum(model.predict(X_valid), 0)
    results[model_name] = {
        "mae": float(mean_absolute_error(y_valid, valid_pred)),
        "rmse": float(math.sqrt(mean_squared_error(y_valid, valid_pred))),
        "rmsle": rmsle(y_valid, valid_pred),
        "r2": float(r2_score(y_valid, valid_pred)),
    }
    trained_models[model_name] = model

metrics_df = pd.DataFrame(results).T.sort_values("rmsle")
metrics_df


## Refit Best Model and Save Artifacts

In [ ]:
best_model_name = metrics_df.index[0]
best_model = candidate_models[best_model_name]
best_model.fit(X, y)

metrics_payload = {
    "competition": "playground-series-s5e5",
    "target": TARGET_COLUMN,
    "best_model_name": best_model_name,
    "validation_metrics": results,
    "selected_metric": "rmsle",
    "random_state": RANDOM_STATE,
    "train_rows": int(len(train_df)),
    "feature_count": int(len(feature_columns)),
}

model_path = ARTIFACT_DIR / "calorie_expenditure_model.joblib"
metrics_path = ARTIFACT_DIR / "metrics.json"
schema_path = ARTIFACT_DIR / "feature_schema.json"

joblib.dump(best_model, model_path)
metrics_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
schema_path.write_text(json.dumps(feature_schema, indent=2), encoding="utf-8")

print(f"Best model: {best_model_name}")
print(f"Saved model: {model_path}")
print(f"Saved metrics: {metrics_path}")
print(f"Saved schema: {schema_path}")
metrics_payload


## Create Submission

In [ ]:
if test_df is not None:
    missing_features = [column for column in feature_columns if column not in test_df.columns]
    if missing_features:
        raise ValueError(f"Test set is missing features: {missing_features}")

    test_predictions = np.maximum(best_model.predict(test_df[feature_columns]), 0)
    submission_id_column = id_columns[0] if id_columns and id_columns[0] in test_df.columns else None
    submission_df = pd.DataFrame(
        {
            "id": test_df[submission_id_column] if submission_id_column else np.arange(len(test_df)),
            TARGET_COLUMN: test_predictions,
        }
    )
    submission_path = ARTIFACT_DIR / "submission.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"Saved submission: {submission_path}")
    display(submission_df.head())
else:
    print("No test.csv found, so submission.csv was not created.")


## Quick Inference Check

In [ ]:
loaded_model = joblib.load(model_path)
sample_predictions = np.maximum(loaded_model.predict(X.head(5)), 0)
pd.DataFrame({"actual": y.head(5).to_numpy(), "predicted": sample_predictions})
